In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import other libraries if needed

In [ ]:
df = pd.read_csv("train.csv")
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_boxplots_grid(dataframe, n_cols=3, subplot_size=(5, 3)):
    """
    Plots a grid of boxplots for all numerical non-binary columns in a DataFrame.

    :param dataframe: pandas DataFrame to plot.
    :param n_cols: Number of columns in the grid.
    :param subplot_size: Size of each subplot (width, height).
    """
    # Identify numerical columns that are not binary (more than 2 unique values)
    numerical_cols = [col for col in dataframe.columns if dataframe[col].dtype != 'object' and dataframe[col].nunique() > 2]

    # Determine the grid size (number of rows and columns in the grid)
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols  # Computes the required number of rows

    # Create a figure with subplots in a grid
    plt.figure(figsize=(subplot_size[0] * n_cols, subplot_size[1] * n_rows))
    for i, col in enumerate(numerical_cols):
        plt.subplot(n_rows, n_cols, i + 1)  # Create a subplot for each column
        sns.boxplot(y=dataframe[col])
        plt.title(col)
    plt.tight_layout()  # Adjusts the subplots to fit in the figure area
    plt.show()

In [ ]:
plot_boxplots_grid(df)

In [ ]:
# Plot the distribution for target column
import matplotlib.pyplot as plt

def plot_churn_distribution(data, column='Churn Label'):
    """
    Plots the distribution of the churn label in a pie chart.

    :param data: DataFrame containing the customer data.
    :param column: Name of the column containing the churn label.
    """
    # Calculating the distribution
    churn_dist = data[column].value_counts()

    # Plotting
    plt.figure(figsize=(8, 6))
    plt.pie(churn_dist, labels=churn_dist.index, autopct='%1.1f%%', startangle=140)
    plt.title('Distribution of Churn Label')
    plt.show()

# Testing the function with the customer data
plot_churn_distribution(df)

In [ ]:
# Plot the distribution for each numerical columns based on the churn label

import seaborn as sns

def plot_numerical_distribution_by_churn(data, churn_column='Churn Label'):
    """
    Plots the distribution of each numerical column in the data, separated by churn label.

    :param data: DataFrame containing the customer data.
    :param churn_column: Name of the column containing the churn label.
    """
    # Identifying numerical columns
    numerical_columns = data.select_dtypes(include=['number']).columns

    # Number of rows and columns for the subplot
    n_cols = 2
    n_rows = (len(numerical_columns) + 1) // n_cols

    # Creating subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten()

    # Plotting each numerical column
    for i, col in enumerate(numerical_columns):
        sns.histplot(data=data, x=col, hue=churn_column, ax=axes[i], kde=True, element='step')
        axes[i].set_title(f'Distribution of {col} by Churn Label')
        axes[i].set_ylabel('Count')

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [ ]:
plot_numerical_distribution_by_churn(df[['Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV', 'Churn Label']])

In [ ]:
# Plot correlation matrix
corr = df.drop('CustomerID', axis=1).corr()

# Plot heatmap
plt.figure(figsize=(8, 8))
sns.heatmap(corr, annot=True, square=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation matrix')
plt.show()

In [ ]:
# Split training set and validation set here, store into variables train_set and val_set.
# Remember to also keep the original training set before splitting. This will come important later.

# Split the data into training and validation sets
from sklearn.model_selection import train_test_split

train_set, val_set = train_test_split(df, test_size=0.2, random_state=42)
# test_size = 0.2 artinya 80% adalah training set dan 20% adalah validation set

In [ ]:
train_set.shape, val_set.shape

In [ ]:
# Drop the columns

columns_to_drop = ['CustomerID', 'Latitude', 'Longitude']

train_set.drop(columns=columns_to_drop, inplace=True)
val_set.drop(columns=columns_to_drop, inplace=True)

train_set.head()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureCreator(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        
        # Explore ways to create or transform new features!

        # Create a new feature called 'Total Charges' by multiplying 'Monthly Charges' and 'Tenure Months'
        # diganti sesuai performance, tidak ada penambahan kolom atau ditambahkan kolomnya
        X['Total Charges'] = X['Monthly Charges'] * X['Tenure Months']

        return X

In [ ]:
# Import necessary scalers
from sklearn.preprocessing import StandardScaler

class FeatureScaler(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        
        # Fit the scaler here
        # mengukur dan menyimpan rumus/ ukuran data
        # jangan fitting ke validation set

        self.numerical_cols = [col for col in X.columns if X[col].dtype != 'object' and X[col].nunique() > 2]
        self.sc = StandardScaler().fit(X[self.numerical_cols])
        
        return self
    
    def transform(self, X):
        
        # Perform scaling here

        X[self.numerical_cols] = self.sc.transform(X[self.numerical_cols])
        
        return X

In [ ]:
class FeatureEncoder(BaseEstimator, TransformerMixin):
        
    def fit(self, X, y=None):
        
        return self
    
    def transform(self, X):
        X_encoded = X.copy()
        
        # Encode the categorical variables here
        
        cat_cols = X.select_dtypes(include=['object'])
        
        for col in cat_cols:
            
            # Check if the column is binary
            # label encoded
            if X[col].nunique() == 2 and col != 'Gender':
                
                # Map the column with a dictionary
                X_encoded[col] = X_encoded[col].map({'Yes': 1, 'No': 0})
            
            # one-hot encoded
            else:
                # Perform one-hot encoding
                X_encoded = pd.concat([X_encoded, pd.get_dummies(X_encoded[col], prefix=col)], axis=1)

                # Drop original categorical columns from DataFrame
                X_encoded.drop(col, axis=1, inplace=True)
        
        return X_encoded

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([("featurecreator", FeatureCreator()),
                 ("scaler", FeatureScaler()),
                 ("encoder", FeatureEncoder())])

In [ ]:
# Your code should work up until this point
train_set = pipe.fit_transform(train_set)

# jangan sampe di FIT, hanya transform saja
val_set = pipe.transform(val_set)

In [ ]:
# Import the model
from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
# from catboost import CatBoostClassifier

# Create the model
rf = RandomForestClassifier(random_state=42)
# xgb = XGBClassifier(random_state=42)
# cat = CatBoostClassifier(random_state=42, verbose=False)

In [ ]:
# Split the data into features and target
X_train = train_set.drop('Churn Label', axis=1)
y_train = train_set['Churn Label']

X_val = val_set.drop('Churn Label', axis=1)
y_val = val_set['Churn Label']

# Train the model
rf.fit(X_train, y_train)
# xgb.fit(X_train, y_train)
# cat.fit(X_train, y_train)

In [ ]:
# Import the evaluation metrics
from sklearn.metrics import classification_report

# Make predictions
rf_pred = rf.predict(X_val)
# xgb_pred = xgb.predict(X_val)
# cat_pred = cat.predict(X_val)

# Print classification report for each model
print("Random Forest Classifier")
print(classification_report(y_val, rf_pred))
# print()
# print("XGBoost Classifier")
# print(classification_report(y_val, xgb_pred))
# print()
# print("CatBoost Classifier")
# print(classification_report(y_val, cat_pred))

In [ ]:
import sys
print(sys.version)